# BreizhCrops %10 Pilot Çalışması

**Öğrenci:** Esmanur Polattimur  
**Numara:** 241230014

Bu notebook büyük BreizhCrops veri setinin ön işleme adımlarını öğrenci anlatımıyla göstermektedir. Pilot çalışma nihai sonuç değildir.

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER VE DOSYA YOLLARI
# ============================================================

from pathlib import Path
import subprocess
import sys

import h5py
import pandas as pd
import matplotlib.pyplot as plt

proje_klasoru = Path.cwd()
veri_klasoru = Path(r"D:\Datasets\BreizhCrops\2017\L1C")
indeks_klasoru = proje_klasoru / "data" / "phase1" / "breizhcrops"
print("Veri klasörü:", veri_klasoru)

In [ ]:
# ============================================================
# 2. DOSYALARI KONTROL ETME
# ============================================================

bolgeler = ["frh01", "frh02", "frh03", "frh04"]
for bolge in bolgeler:
    dosya = veri_klasoru / (bolge + ".h5")
    boyut_gb = dosya.stat().st_size / (1024 ** 3)
    print(bolge.upper(), "var mı:", dosya.is_file(), "| GB:", round(boyut_gb, 2))

In [ ]:
# ============================================================
# 3. SINIF VE BÖLGE DAĞILIMI
# ============================================================

esleme = pd.read_csv(indeks_klasoru / "classmapping.csv")
esleme = esleme[["id", "classname", "code"]]
esleme = esleme.rename(columns={"id": "class_id", "classname": "class_name"})

indeks_listesi = []
for bolge in bolgeler:
    indeks = pd.read_csv(indeks_klasoru / (bolge + ".csv"))
    indeks = indeks.merge(esleme, left_on="CODE_CULTU", right_on="code", how="inner")
    indeks["region"] = bolge.upper()
    indeks_listesi.append(indeks)

tum_indeksler = pd.concat(indeks_listesi, ignore_index=True)
print("Kullanılabilir toplam parsel:", len(tum_indeksler))
pd.crosstab(tum_indeksler["class_name"], tum_indeksler["region"])

## Veri ayrımı ve ön işleme

FRH01 ve FRH02 eğitim, FRH03 doğrulama, FRH04 final test olarak ayrılır. QA60 ile bulut kontrolü yapılır. Bantlar 10000'e bölünür; NDVI, NDRE ve NDMI hesaplanır. Yıl 24 yarım aylık döneme ayrılır ve fenolojik özellikler çıkarılır.

In [ ]:
# ============================================================
# 4. TEK BİR PARSELİ OKUMA VE NDVI HESAPLAMA
# ============================================================

ornek = tum_indeksler.iloc[0]
h5_dosyasi = veri_klasoru / (ornek["region"].lower() + ".h5")
with h5py.File(h5_dosyasi, "r") as h5:
    dizi = h5[ornek["path"]][:]

sutunlar = ["B1", "B10", "B11", "B12", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "QA10", "QA20", "QA60", "tarih"]
ornek_df = pd.DataFrame(dizi, columns=sutunlar)
ornek_df["tarih"] = pd.to_datetime(ornek_df["tarih"], unit="ns")
ornek_df["NDVI"] = (ornek_df["B8"] - ornek_df["B4"]) / (ornek_df["B8"] + ornek_df["B4"])
print("Parsel:", ornek["id"], "| Boyut:", ornek_df.shape)
ornek_df.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ornek_df["tarih"], ornek_df["NDVI"], marker="o")
plt.title("Örnek Parselin NDVI Zaman Serisi")
plt.xlabel("Tarih")
plt.ylabel("NDVI")
plt.grid(alpha=0.3)
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5. HIZLI KOD TESTİ
# ============================================================

# Her bölge ve sınıftan en fazla 2 parsel kullanılır.
komut = [sys.executable, str(proje_klasoru / "run_breizh_pilot.py"), "--hizli-test", "--uzerine-yaz"]
subprocess.run(komut, cwd=proje_klasoru, check=True)

In [ ]:
# ============================================================
# 6. HIZLI TEST ÇIKTISI
# ============================================================

ozellik_dosyasi = proje_klasoru / "output" / "phase3" / "smoke_features.csv.gz"
ozellik_df = pd.read_csv(ozellik_dosyasi)
print("Özellik tablosu boyutu:", ozellik_df.shape)
ozellik_df[["parcel_id", "region", "class_name", "valid_observation_count"]].head()

In [ ]:
# ============================================================
# 7. %10 PİLOTU ÇALIŞTIRMA
# ============================================================

TAM_PILOTU_CALISTIR = False
if TAM_PILOTU_CALISTIR:
    komut = [sys.executable, str(proje_klasoru / "run_breizh_pilot.py"), "--oran", "0.10", "--uzerine-yaz"]
    subprocess.run(komut, cwd=proje_klasoru, check=True)
else:
    print("%10 pilot henüz başlatılmadı. Hızlı test başarıyla tamamlandı.")